In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [5]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="mbazaNLP/kinyarwanda-tts-dataset", 
#     repo_type="dataset", local_dir="./kinyarwanda-tts-dataset")



In [6]:
!ls kinyarwanda-tts-dataset

LICENSE  README.md  audio  tts-dataset.csv


In [11]:
rows = pd.read_csv('kinyarwanda-tts-dataset/tts-dataset.csv', header = None, sep = ' "').to_dict(orient = 'records')
rows[0]

/usr/lib/python3/dist-packages/pandas/util/_decorators.py:311: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  return func(*args, **kwargs)


{0: 'TTS_1_2',
 1: 'ntivuga ko nehemiya yakoreye umugabo wa esiteri umwami ahasuwerusi ahubwo yakoreye uwamusimbuye"'}

In [17]:
base = 'kinyarwanda-tts-dataset_audio'
!mkdir {base}

In [27]:
def loop(rows):
    rows, _ = rows
    data = []
    for row in tqdm(rows):
        f = os.path.join('kinyarwanda-tts-dataset/audio', row[0]).replace('TTS_', 'TTS ') + '.wav'
        if not os.path.exists(f):
            continue

        t = row[1].strip()[:-1]
        if len(t) < 2:
            continue
        
        audio_filename = f.replace('/', '-').replace('.wav', '.mp3').replace(' ', '-')
        audio_filename = os.path.join(base, audio_filename)

        audio_np, sr = sf.read(f)
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)

        data.append({
            'audio_filename': audio_filename,
            'text': t,
            'speaker': f"{base}"
        })
    return data

In [28]:
data = loop((rows[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 10.80it/s]


In [30]:
data = multiprocessing(rows, loop, cores = 20)

  1%|          | 2/199 [00:00<00:17, 11.47it/s]

100%|██████████| 199/199 [00:24<00:00,  8.11it/s]


In [31]:
len(data)

3992

In [32]:
audio_files = [d['audio_filename'] for d in data]

with open('kinyarwanda-tts-dataset-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [33]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'kinyarwanda-tts-dataset_audio/kinyarwanda-tts-dataset-audio-TTS-1_2.mp3',
 'text': 'ntivuga ko nehemiya yakoreye umugabo wa esiteri umwami ahasuwerusi ahubwo yakoreye uwamusimbuye',
 'speaker': 'kinyarwanda-tts-dataset_audio'}

In [34]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'kinyarwanda-tts-dataset')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 458.19ba/s]
Processing Files (1 / 1): 100%|██████████|  267kB /  267kB, 1.34MB/s  
New Data Upload: 100%|██████████|  267kB /  267kB, 1.34MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.94 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/6027d35c6cbc7925cdfb19ac443ebcdf7acff3e0', commit_message='Upload dataset', commit_description='', oid='6027d35c6cbc7925cdfb19ac443ebcdf7acff3e0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [36]:
# !zip -rq kinyarwanda-tts-dataset_audio.zip kinyarwanda-tts-dataset_audio
# !hf upload malaysia-ai/Multilingual-TTS kinyarwanda-tts-dataset_audio.zip --repo-type=dataset